<a href="https://colab.research.google.com/github/kumarsirish/FDP-AGENENTIC-AI-RAG/blob/main/rag-adk-06/professor_assistant_adk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Professor's AI Assistant — Google ADK Multi-Agent

| Agent | Job |
|---|---|
| 🗂️ **Syllabus Agent** | Answers questions about your syllabus |
| 📅 **Timetable Agent** | Reads your timetable, finds free slots |
| 📝 **Quiz Agent** | Generates quiz questions and MCQs |

**Setup:** Add your Gemini key to Colab Secrets as `GEMINI_API_KEY` (🔑 icon in sidebar).
Run Cell 1, **restart runtime**, then run all remaining cells.

## Cell 1 — Install
> ⚠️ Restart runtime after this cell.

In [ ]:
!pip install -q google-adk PyPDF2
print('✅ Done — restart runtime now.')

## Cell 2 — Imports & API Key

In [ ]:
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')

from google.adk.agents import LlmAgent
from google.adk.tools.agent_tool import AgentTool
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import ToolContext
from google.genai.types import Content, Part
print('✅ Ready.')

## Cell 3 — Upload Files (optional)
Upload your **syllabus** and/or **timetable** as PDF. Cancel to skip.

In [ ]:
import PyPDF2, io
import re, gdown # Added for regex to extract Google Drive file ID

# Google Drive URLs for the provided files
syllabus_drive_url = 'https://drive.google.com/file/d/1K26k6ejc1MKzw7HegTUw6kxatj9BQmBv/view?usp=drive_link'
timetable_drive_url = 'https://drive.google.com/file/d/1v136u307B-zTp-S_jRg0kK8x7S3SprGJ/view?usp=drive_link'
SYLLABUS_TEXT  = ''
TIMETABLE_TEXT = ''

def extract_pdf(content):
    return '\n'.join(p.extract_text() or '' for p in PyPDF2.PdfReader(io.BytesIO(content)).pages)

def download_file_from_google_drive(file_id, destination):
    gdown.download(id=file_id, output=destination, quiet=True)


def get_drive_file_id(drive_url):
    """Extracts the Google Drive file ID from a given URL."""
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', drive_url)
    if match:
        return match.group(1)
    raise ValueError(f"Could not extract Google Drive file ID from URL: {drive_url}")

print('📄 Downloading SYLLABUS PDF from Google Drive')
try:
    syllabus_id = get_drive_file_id(syllabus_drive_url)
    download_file_from_google_drive(syllabus_id, 'syllabus.pdf') # Call function from 2fd64a67
    with open('syllabus.pdf', 'rb') as f:
        SYLLABUS_TEXT = extract_pdf(f.read())
    print(f'✅ Syllabus loaded ({len(SYLLABUS_TEXT)} chars)')
except Exception as e:
    print(f'⚠️ Skipped Syllabus download. Error: {e}')

print('\n🗓️ Downloading TIMETABLE PDF from Google Drive')
try:
    timetable_id = get_drive_file_id(timetable_drive_url)
    download_file_from_google_drive(timetable_id, 'timetable.pdf') # Call function from 2fd64a67
    with open('timetable.pdf', 'rb') as f:
        TIMETABLE_TEXT = extract_pdf(f.read())
    print(f'✅ Timetable loaded ({len(TIMETABLE_TEXT)} chars)')
except Exception as e:
    print(f'⚠️ Skipped Timetable download. Error: {e}')

## Cell 4 — Tools
> 📌 Tools are plain Python functions. The **docstring** is what the LLM reads to decide when to call each tool.

In [ ]:
def get_syllabus(tool_context: ToolContext) -> str:
    """Return the full uploaded syllabus text."""
    text = tool_context.state.get('syllabus', SYLLABUS_TEXT)
    return text if text.strip() else 'No syllabus uploaded.'

def search_syllabus(topic: str, tool_context: ToolContext) -> str:
    """Search the syllabus for a topic or keyword.
    Args: topic: keyword to search for.
    """
    text = tool_context.state.get('syllabus', SYLLABUS_TEXT)
    if not text.strip(): return 'No syllabus uploaded.'
    matches = [l for l in text.splitlines() if topic.lower() in l.lower()]
    return ('\n'.join(matches[:20])) if matches else f"'{topic}' not found."

def get_timetable(tool_context: ToolContext) -> str:
    """Return the full timetable as structured lines (Day | Time | Subject | Professor)."""
    text = tool_context.state.get('timetable', TIMETABLE_TEXT)
    if not text.strip(): return 'No timetable uploaded.'
    lines = [l for l in text.splitlines() if ' | ' in l]
    return '\n'.join(lines) if lines else text

def search_timetable(query: str, tool_context: ToolContext) -> str:
    """Search the timetable by professor name, day, or subject.
    Use this when the user mentions a name (e.g. 'kumar', 'patel'),
    a day (e.g. 'Monday'), or a subject (e.g. 'Machine Learning').
    Each result line format: Day | Time | Subject | Professor
    Args: query: name, day, or subject to search for.
    """
    text = tool_context.state.get('timetable', TIMETABLE_TEXT)
    if not text.strip(): return 'No timetable uploaded.'
    lines = [l for l in text.splitlines() if ' | ' in l]
    matches = [l for l in lines if query.lower() in l.lower()]
    return '\n'.join(matches) if matches else f"No entries found for '{query}'."


def get_free_slots(professor_name: str, tool_context: ToolContext) -> str:
    """Compute free time slots for a professor across the whole week.
    ALWAYS use this tool when asked about free time, office hours, or availability.
    Args: professor_name: full or partial name e.g. 'kumar', 'patel', 'rajesh'.
    """
    text = tool_context.state.get('timetable', TIMETABLE_TEXT)
    if not text.strip(): return 'No timetable uploaded.'

    all_slots = ['9:00-10:00', '10:00-11:00', '11:00-12:00',
                 '2:00-3:00', '3:00-4:00', '4:00-5:00']
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    lines = [l for l in text.splitlines() if ' | ' in l]
    busy = {}
    for line in lines:
        parts = [p.strip() for p in line.split('|')]
        if len(parts) == 4:
            day, slot, subject, prof = parts
            if professor_name.lower() in prof.lower() and 'Break' not in subject:
                busy.setdefault(day, []).append(slot)

    if not any(busy.values()):
        return f"No professor matching '{professor_name}' found in timetable."

    result = [f"Schedule for professor matching '{professor_name}':"]
    for day in days:
        busy_slots = busy.get(day, [])
        free_slots = [s for s in all_slots if s not in busy_slots]
        result.append(f"  {day}:")
        result.append(f"    Busy : {', '.join(busy_slots) if busy_slots else 'None'}")
        result.append(f"    Free : {', '.join(free_slots) if free_slots else 'None'}")
    return '\n'.join(result)


def save_quiz(title: str, content: str, tool_context: ToolContext) -> str:
    """Save a generated quiz to session memory.
    Args: title: quiz title. content: full quiz text.
    """
    quizzes = tool_context.state.get('quizzes', [])
    quizzes.append({'title': title, 'content': content})
    tool_context.state['quizzes'] = quizzes
    return f"Saved '{title}'. Total: {len(quizzes)}."

def list_quizzes(tool_context: ToolContext) -> str:
    """List all quizzes saved in this session."""
    q = tool_context.state.get('quizzes', [])
    return '\n'.join(f"{i+1}. {x['title']}" for i,x in enumerate(q)) if q else 'No quizzes saved.'

print('✅ Tools defined.')

## Cell 5 — Agents
> 📌 Each agent's **`description`** tells the orchestrator when to delegate to it.

In [ ]:
MODEL = 'gemini-2.5-flash'

syllabus_agent = LlmAgent(
    name='syllabus_agent',
    model=MODEL,
    description='Answers questions about the course syllabus — topics, weeks, content.',
    instruction='Use get_syllabus() or search_syllabus(topic) to answer.',
    tools=[get_syllabus, search_syllabus],
)

timetable_agent = LlmAgent(
    name='timetable_agent',
    model=MODEL,
    description='Answers schedule questions — class times, professor schedules, free slots. Also handles queries by professor name.',
    instruction=(
        'Use the right tool for each question:\n'
        '- Free time / office hours / availability → get_free_slots(professor_name)\n'
        '- Schedule for a name or day → search_timetable(query)\n'
        '- Full timetable → get_timetable()\n'
        'Never infer free slots yourself — always call get_free_slots().'
    ),
    tools=[get_timetable, search_timetable, get_free_slots],
)

quiz_agent = LlmAgent(
    name='quiz_agent',
    model=MODEL,
    description='Generates quiz questions and MCQs on any topic or from the syllabus.',
    instruction='Generate questions with answer keys. Save each quiz with save_quiz().',
    tools=[save_quiz, list_quizzes, get_syllabus],
)

# Orchestrator — wraps sub-agents as AgentTool objects
root_agent = LlmAgent(
    name='professor_assistant',
    model=MODEL,
    description="Professor's assistant for syllabus, timetable, and quiz tasks.",
    instruction=(
        'Route to the right agent:\n'
        '- syllabus_agent : syllabus content, topics\n'
        '- timetable_agent: schedule, free slots\n'
        '- quiz_agent     : generate quizzes, MCQs'
    ),
    tools=[
        AgentTool(agent=syllabus_agent),
        AgentTool(agent=timetable_agent),
        AgentTool(agent=quiz_agent),
    ],
)
print('✅ Agents ready.')

## Cell 6 — Runner, Session & ask()
> `ask()` shows every delegation step (→ agent called) and the final answer.

In [ ]:
APP_NAME = 'prof_assistant'
USER_ID  = 'professor'

session_service = InMemorySessionService()
runner  = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

if SYLLABUS_TEXT:  session.state['syllabus']  = SYLLABUS_TEXT
if TIMETABLE_TEXT: session.state['timetable'] = TIMETABLE_TEXT

def ask(question):
    """Send a question. Prints each agent delegation step and the final answer."""
    print(f'\n🧑\u200d🏫 {question}\n' + '─'*55)
    msg = Content(role='user', parts=[Part(text=question)])
    for event in runner.run(user_id=USER_ID, session_id=session.id, new_message=msg):
        if not event.content: continue
        for part in event.content.parts:
            if hasattr(part, 'function_call') and part.function_call:
                print(f'  → [{part.function_call.name}]')
            elif hasattr(part, 'text') and part.text and event.is_final_response():
                print(part.text.strip())

print('✅ Ready.')

---
## Demo — Syllabus

In [ ]:
ask('Summarise the course syllabus.')

In [ ]:
ask('Is github covered? Which section?')

## Demo — Timetable

In [ ]:
ask('I am Dr. Rajesh Kumar. What is my schedule on Monday?')

In [ ]:
ask('When do I have free time for office hours?')

## Demo — Quiz

In [ ]:
ask('Create a 2-question MCQ quiz based on the devops syllabus.')

## Demo — Multi-agent chaining
One question, two agents called in sequence.

In [ ]:
ask('Read my syllabus and create a quiz on the first topic. List the Quiz')